In [1]:
pip install --upgrade huggingface_hub



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import re

# 2.1 Style exemplars for your narrator
STYLE_EXEMPLARS = [
    "“You know, when I talk about hacking the brain, I’m half-joking—but it’s also terrifyingly real.”",
    "“Imagine coffee that reads your mind before you even open your eyes—sounds like sci-fi, right?”",
    "“Let’s be honest: we all hit snooze too many times—just me? Probably not.”"
]

# 2.2 Prompt template that tells DeepSeek-R1 to preserve style
PROMPT_TEMPLATE = """
Below is a paragraph from a narrator who speaks in his unique specifical style.
Here are a few exemplar sentences of that style:

1. {ex0}
2. {ex1}
3. {ex2}

Translate the following English text into Russian, preserving that same informal, engaging tone,
including any humor or rhetorical flourishes where possible to fully convey the emotional and stylistic component:

--- Begin English Text ---
{chunk}
--- End English Text ---

Provide only the Russian translation (no extra commentary).
""".strip()

# 2.3 Chunking helper: split a long string into ≤ max_chars chunks at sentence boundaries
def chunk_text_for_llm(text: str, max_chars: int = 3000) -> list[str]:
    """
    Splits `text` into chunks no longer than `max_chars` characters,
    breaking at sentence-ending punctuation (.!?).
    """
    text = text.strip()
    if len(text) <= max_chars:
        return [text]

    sentence_end_re = re.compile(r'(?<=[\.\?\!])\s+')
    sentences = sentence_end_re.split(text)
    chunks = []
    current = ""
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        # If adding this sentence would exceed max_chars, flush the current chunk
        if len(current) + len(sent) + 1 > max_chars:
            if current:
                chunks.append(current)
            current = sent
        else:
            current = (current + " " + sent).strip() if current else sent

    if current:
        chunks.append(current)
    return chunks


In [4]:
import os
import time
from huggingface_hub import InferenceClient

# 3.1 Initialize InferenceClient once (reads API key from environment)
client = InferenceClient(api_key=os.getenv("HUGGINGFACE_API_KEY"))

def translate_with_deepseek_r1(
    full_transcript: str,
    style_exemplars: list[str],
    max_chars_per_chunk: int = 2500,
    temperature: float = 0.7,
    max_retries: int = 2
) -> str:
    """
    Splits `full_transcript` into smaller chunks, prompts DeepSeek-R1 to translate each chunk into Russian
    preserving the narrator’s style, and returns the concatenated Russian text.
    """
    chunks = chunk_text_for_llm(full_transcript, max_chars=max_chars_per_chunk)
    russian_pieces = []

    for idx, chunk in enumerate(chunks):
        # 3.2 Build the prompt
        prompt = PROMPT_TEMPLATE.format(
            ex0=style_exemplars[0],
            ex1=style_exemplars[1],
            ex2=style_exemplars[2],
            chunk=chunk
        )

        # 3.3 Attempt the API call up to max_retries times
        for attempt in range(max_retries):
            try:
                response = client.chat.completions.create(
                    model="deepseek-ai/DeepSeek-R1",
                    messages=[
                        {"role": "system", "content": "You are a translation assistant that preserves style."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=temperature,
                    max_tokens=1500
                )
                rus_chunk = response.choices[0].message.content.strip()
                russian_pieces.append(rus_chunk)
                break  # success → exit retry loop

            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(1 + attempt)  # simple backoff
                    continue
                else:
                    raise RuntimeError(f"DeepSeek-R1 API failed on chunk {idx}: {e}")

    # 3.4 Join all Russian pieces with double newlines to preserve paragraph breaks
    return "\n\n".join(russian_pieces)


/home/alexander/Documents/DLS-Speech-Translation-Project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import re
from typing import List, Tuple

# 1) Define a small set of colloquial markers / style keywords
COLLOQUIAL_MARKERS = [
    r"\byou know\b", r"\blet's\b", r"\bI mean\b", r"\bright\b",
    r"\bI'm\b", r"\bwe're\b", r"\bain't\b", r"\bkind of\b", r"\bsort of\b"
]
FIRST_PERSON = [r"\bI\b", r"\bme\b", r"\bmy\b", r"\bwe\b", r"\bour\b"]
SECOND_PERSON = [r"\byou\b", r"\byour\b"]


def split_into_sentences(text: str) -> List[str]:
    """
    A simple regex‐based sentence splitter. Splits on period, exclamation, or question mark
    followed by whitespace and a capital letter. Falls back on newlines if needed.
    """
    # First, normalize whitespace
    text = text.strip().replace("\n", " ")
    # Use lookahead to split on . ! ? when followed by a space + capital letter
    sentence_end_re = re.compile(r"(?<=[\.\?\!])\s+(?=[A-Z])")
    sentences = sentence_end_re.split(text)

    # If we got nothing (e.g. no punctuation), fall back to splitting on .!? directly
    if len(sentences) == 1:
        sentences = re.split(r"(?<=[\.\?\!])\s+", text)

    # Trim whitespace on each
    return [s.strip() for s in sentences if s.strip()]


def score_sentence(sent: str) -> int:
    """
    Assign a simple score to `sent` based on presence of rhetorical/stylistic markers:
      +2 if it contains a "?" (rhetorical / energetic)
      +1 if it contains a "!" (exclamation)
      +1 per colloquial marker (e.g. "you know", "let's", "I mean")
      +1 per first‐person usage (I, we, my, etc.)
      +1 per second‐person usage (you, your)
    """
    score = 0

    # Count question marks
    if "?" in sent:
        score += 2

    # Count exclamation points
    if "!" in sent:
        score += 1

    # Check colloquial markers
    for pattern in COLLOQUIAL_MARKERS:
        if re.search(pattern, sent, flags=re.IGNORECASE):
            score += 1

    # Check first‐person pronouns
    for pattern in FIRST_PERSON:
        if re.search(pattern, sent):
            score += 1

    # Check second‐person pronouns
    for pattern in SECOND_PERSON:
        if re.search(pattern, sent):
            score += 1

    return score


def extract_style_exemplars(
    transcript: str,
    num_exemplars: int = 3
) -> List[str]:
    """
    From `transcript`, pick the top `num_exemplars` sentences most likely to reflect the narrator’s style.
    Returns a list of sentences (strings), in descending order of “style‐score.”
    """
    # 1) Split into sentences
    sentences = split_into_sentences(transcript)

    # 2) Score each sentence
    scored: List[Tuple[int, str]] = []
    for sent in sentences:
        s = score_sentence(sent)
        scored.append((s, sent))

    # 3) Sort by descending score, then by length (longer = more content)
    scored.sort(key=lambda x: (x[0], len(x[1])), reverse=True)

    # 4) Take the top `num_exemplars`, but skip any that are too short (< 20 chars)
    exemplars: List[str] = []
    for score, sent in scored:
        if len(sent) < 20:
            continue
        exemplars.append(sent)
        if len(exemplars) >= num_exemplars:
            break

    # If we didn’t find enough, just pad with the first few sentences
    if len(exemplars) < num_exemplars:
        for sent in sentences:
            if sent not in exemplars and len(sent) >= 20:
                exemplars.append(sent)
            if len(exemplars) >= num_exemplars:
                break

    return exemplars[:num_exemplars]


In [11]:
# from translation_helpers import extract_style_exemplars

# Suppose `full_transcript` is the string you got from ASR (11-minute lecture, for example).
full_transcript = """
Hello everyone! You know, when I talk about hacking the brain, I’m actually being quite serious—though it sounds like a joke.
Imagine coffee that reads your mind before you even open your eyes—sounds like sci-fi, right?
Let’s be honest: most diets fail because we all cheat a little too often.
In this talk, we’ll explore how algorithms can predict emotions.
We’re going to dive into both the tech and the ethics, so stay tuned.
Feel free to ask questions anytime—no question is too dumb!
"""

# Extract 3 exemplars
exemplars = extract_style_exemplars(full_transcript, num_exemplars=3)
print("Style exemplars:")
for e in exemplars:
    print("-", e)


Style exemplars:
- Imagine coffee that reads your mind before you even open your eyes—sounds like sci-fi, right?
- You know, when I talk about hacking the brain, I’m actually being quite serious—though it sounds like a joke.
- Let’s be honest: most diets fail because we all cheat a little too often.


In [16]:
trans = translate_with_deepseek_r1(full_transcript,exemplars)


In [17]:
idx = trans.index("</think>\n")

In [22]:
print(trans[idx+9:])

Всем привет! Вот знаете, когда я говорю про взлом мозга, я абсолютно серьёзен — хоть это и звучит как шутка.  
Представьте кофе, который читает ваши мысли, ещё до того, как вы открыли глаза — прям как в научной фантастике, правда?  
Давайте честно: большинство диет проваливается, потому что мы все немножко мухлюем слишком часто.  
Сегодня мы разберёмся, как алгоритмы могут предсказывать эмоции.  
Мы нырнём и в технологии, и в этику, так что оставайтесь на связи.  
Не стесняйтесь задавать вопросы в любое время — тупых вопросов не существует!
